In [1]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType

In [2]:
spark = SparkSession.builder \
    .appName("application_bronze_silver_gold_pipeline") \
    .getOrCreate()

print(spark)

In [3]:
from pathlib import Path

# Resolve project root (../ from this notebook file) so relative paths work no matter CWD
project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

raw_file_path = project_root / "data" / "raw" / "application_data.csv"

bronze_output_path = project_root / "data" / "bronze" / "bronze_application_raw.parquet"
silver_output_path = project_root / "data" / "silver" / "silver_application_clean.parquet"

gold_model_base_path = project_root / "data" / "gold" / "gold_application_model_base.parquet"
gold_dev_path = project_root / "data" / "gold" / "gold_development_dataset.parquet"
gold_oot_path = project_root / "data" / "gold" / "gold_final_oot_dataset.parquet"
gold_window_def_path = project_root / "data" / "gold" / "gold_rolling_window_definition.parquet"

print("project_root:", project_root)
print("raw_file_path:", raw_file_path)

project_root: c:\Users\88696\Desktop\ml_project
raw_file_path: c:\Users\88696\Desktop\ml_project\data\raw\application_data.csv


In [4]:
key_col = "案件編號"
date_col = "進件日"
label_col = "授信結果"

candidate_feature_cols = [
    "申辦期數",
    "原申辦金額",
    "性別",
    "年齡",
    "教育程度",
    "婚姻狀況",
    "月所得",
    "車齡",
    "職業說明",
    "居住地",
    "所留市內電話數",
    "內部往來次數",
    "近半年同業查詢次數",
    "廠牌車型",
    "動產設定"
]

exclude_from_model_cols = [
    "案件編號",
    "成功案例"
]

In [7]:
df_bronze = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(str(raw_file_path))  # PySpark requires string path, not Path object
)

df_bronze = df_bronze.withColumn("bronze_load_timestamp", F.current_timestamp())

print("Bronze row count:", df_bronze.count())
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

Bronze row count: 184051
root
 |-- 案件編號: string (nullable = true)
 |-- 進件日: string (nullable = true)
 |-- 申辦期數: string (nullable = true)
 |-- 原申辦金額: string (nullable = true)
 |-- 性別: string (nullable = true)
 |-- 年齡: string (nullable = true)
 |-- 教育程度: string (nullable = true)
 |-- 婚姻狀況: string (nullable = true)
 |-- 月所得: string (nullable = true)
 |-- 車齡: string (nullable = true)
 |-- 職業說明: string (nullable = true)
 |-- 居住地: string (nullable = true)
 |-- 所留市內電話數: string (nullable = true)
 |-- 內部往來次數: string (nullable = true)
 |-- 近半年同業查詢次數: string (nullable = true)
 |-- 廠牌車型: string (nullable = true)
 |-- 動產設定: string (nullable = true)
 |-- 授信結果: string (nullable = true)
 |-- 成功案例: string (nullable = true)
 |-- bronze_load_timestamp: timestamp (nullable = false)

+----------+--------+--------+----------+----+----+--------+--------+-------------+----+----------------------+------+--------------+------------+------------------+--------+--------+---------+--------+------------------------

In [9]:
df_bronze.write.mode("overwrite").parquet(str(bronze_output_path))
print(f"Bronze saved to: {bronze_output_path}")

Bronze saved to: c:\Users\88696\Desktop\ml_project\data\bronze\bronze_application_raw.parquet


In [91]:
df = spark.read.parquet(str(bronze_output_path))

print("Bronze row count:", df.count())
print(df.columns)

Bronze row count: 184051
['案件編號', '進件日', '申辦期數', '原申辦金額', '性別', '年齡', '教育程度', '婚姻狀況', '月所得', '車齡', '職業說明', '居住地', '所留市內電話數', '內部往來次數', '近半年同業查詢次數', '廠牌車型', '動產設定', '授信結果', '成功案例', 'bronze_load_timestamp']


In [92]:
for c in df.columns:
    df = df.withColumnRenamed(c, c.strip())

print(df.columns)

['案件編號', '進件日', '申辦期數', '原申辦金額', '性別', '年齡', '教育程度', '婚姻狀況', '月所得', '車齡', '職業說明', '居住地', '所留市內電話數', '內部往來次數', '近半年同業查詢次數', '廠牌車型', '動產設定', '授信結果', '成功案例', 'bronze_load_timestamp']


In [93]:
df = df.withColumn(
    "進件日",
    F.to_date(F.col("進件日").cast("string"), "yyyyMMdd")
)

df.select(
    F.min("進件日").alias("min_date"),
    F.max("進件日").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2024-04-01|2026-03-31|
+----------+----------+



In [94]:
int_cols = [
    "申辦期數",
    "年齡",
    "所留市內電話數",
    "內部往來次數",
    "近半年同業查詢次數"
]

# 月所得是字串範圍（如 '20,000~24,999'），不能 cast 成 Double
# 會在後面用序位編碼處理
double_cols = [
    "原申辦金額",
    "車齡"
    # "月所得" - 移除！它是類別型欄位
]

# 先處理字面上的 "NULL" 字串，轉為真正的 null
for c in int_cols + double_cols:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.when(
                (F.upper(F.col(c)) == "NULL") | (F.col(c) == ""),
                None
            ).otherwise(F.col(c))
        )

# 再進行 type casting
for c in int_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(IntegerType()))

for c in double_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(DoubleType()))

df.printSchema()

root
 |-- 案件編號: string (nullable = true)
 |-- 進件日: date (nullable = true)
 |-- 申辦期數: integer (nullable = true)
 |-- 原申辦金額: double (nullable = true)
 |-- 性別: string (nullable = true)
 |-- 年齡: integer (nullable = true)
 |-- 教育程度: string (nullable = true)
 |-- 婚姻狀況: string (nullable = true)
 |-- 月所得: string (nullable = true)
 |-- 車齡: double (nullable = true)
 |-- 職業說明: string (nullable = true)
 |-- 居住地: string (nullable = true)
 |-- 所留市內電話數: integer (nullable = true)
 |-- 內部往來次數: integer (nullable = true)
 |-- 近半年同業查詢次數: integer (nullable = true)
 |-- 廠牌車型: string (nullable = true)
 |-- 動產設定: string (nullable = true)
 |-- 授信結果: string (nullable = true)
 |-- 成功案例: string (nullable = true)
 |-- bronze_load_timestamp: timestamp (nullable = true)



In [95]:
category_cols = [
    "性別",
    "教育程度",
    "婚姻狀況",
    "職業說明",
    "居住地",
    "廠牌車型",
    "動產設定",
    "授信結果",
    "成功案例"
]

for c in category_cols:
    if c in df.columns:
        df = df.withColumn(c, F.trim(F.col(c)))
        df = df.withColumn(
            c,
            F.when((F.col(c) == "") | (F.col(c) == " "), None).otherwise(F.col(c))
        )

In [96]:
df = df.dropna(subset=["案件編號", "進件日", "授信結果"])

print("After dropping missing key/date/label:", df.count())

After dropping missing key/date/label: 184051


In [100]:
window_spec = Window.partitionBy("案件編號").orderBy(F.col("進件日").desc())

df = df.withColumn("rn", F.row_number().over(window_spec))
df = df.filter(F.col("rn") == 1).drop("rn")

print("After deduplication:", df.count())

After deduplication: 184051


In [101]:
df.select(
    F.min("進件日").alias("min_date"),
    F.max("進件日").alias("max_date")
).show()

df.groupBy("授信結果").count().orderBy(F.desc("count")).show(truncate=False)

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2024-04-01|2026-03-31|
+----------+----------+

+----------+------+
|授信結果  |count |
+----------+------+
|APP(核准) |175463|
|WTCD(婉拒)|8588  |
+----------+------+

+----------+------+
|授信結果  |count |
+----------+------+
|APP(核准) |175463|
|WTCD(婉拒)|8588  |
+----------+------+



In [102]:
print("開始進行 Silver standardized 處理")
print("目前筆數:", df.count())

開始進行 Silver standardized 處理
目前筆數: 184051
目前筆數: 184051


In [103]:
df = df.withColumn(
    "授信結果_二元",
    F.when(F.col("授信結果") == "APP(核准)", 1)
     .when(F.col("授信結果") == "WTCD(婉拒)", 0)
     .otherwise(None)
)

df.groupBy("授信結果", "授信結果_二元").count().show()

+----------+-------------+------+
|  授信結果|授信結果_二元| count|
+----------+-------------+------+
|WTCD(婉拒)|            0|  8588|
| APP(核准)|            1|175463|
+----------+-------------+------+



In [104]:
df = df.withColumn(
    "教育程度_是否缺失",
    F.when(F.col("教育程度").isNull(), 1).otherwise(0)
)

In [105]:
df = df.withColumn(
    "教育程度_補值後",
    F.when(
        F.col("教育程度").isNotNull(),
        F.col("教育程度")
    ).when(
        F.col("職業說明").contains("學生(大專生)"),
        F.lit("大學")
    ).when(
        F.col("職業說明").contains("學生(高中職生)"),
        F.lit("高中")
    ).otherwise(F.col("教育程度"))
)

df.groupBy("教育程度_補值後").count().orderBy(F.desc("count")).show(truncate=False)

+---------------+------+
|教育程度_補值後|count |
+---------------+------+
|NULL           |133020|
|大學           |24916 |
|高中           |18598 |
|專科           |4369  |
|國中           |1192  |
|碩士           |1032  |
|其他           |782   |
|博士以上       |142   |
+---------------+------+



In [106]:
fill_missing_cols = ["教育程度", "婚姻狀況", "職業說明", "居住地", "性別", "月所得"]

for c in fill_missing_cols:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.coalesce(F.col(c), F.lit("Missing"))
        )

In [107]:
# 車齡處理：
# -1 = 無資料（缺失）
# > 100 = 輸入錯誤

# 1. 保留原始值
df = df.withColumn("車齡_原始", F.col("車齡"))

# 2. 標記 -1 為缺失
df = df.withColumn(
    "車齡_是否缺失",
    F.when(F.col("車齡") == -1, 1).otherwise(0)
)

# 3. 標記 > 100 為輸入錯誤
df = df.withColumn(
    "車齡_異常旗標",
    F.when(F.col("車齡") > 100, 1).otherwise(0)
)

# 4. 清理後的值：-1 和 > 100 都設為 NULL
df = df.withColumn(
    "車齡_清理後",
    F.when(F.col("車齡") == -1, None)      # 缺失 → NULL
     .when(F.col("車齡") > 100, None)       # 輸入錯誤 → NULL
     .when(F.col("車齡") < 0, None)         # 其他負數也視為異常
     .otherwise(F.col("車齡"))
)

# 檢查結果
print("車齡處理結果：")
df.groupBy("車齡_是否缺失", "車齡_異常旗標").count().show()
df.select("車齡_原始", "車齡_是否缺失", "車齡_異常旗標", "車齡_清理後").show(10, truncate=False)

車齡處理結果：
+-------------+-------------+------+
|車齡_是否缺失|車齡_異常旗標| count|
+-------------+-------------+------+
|            1|            0|171250|
|            0|            0| 12800|
|            0|            1|     1|
+-------------+-------------+------+

+-------------+-------------+------+
|車齡_是否缺失|車齡_異常旗標| count|
+-------------+-------------+------+
|            1|            0|171250|
|            0|            0| 12800|
|            0|            1|     1|
+-------------+-------------+------+

+---------+-------------+-------------+-----------+
|車齡_原始|車齡_是否缺失|車齡_異常旗標|車齡_清理後|
+---------+-------------+-------------+-----------+
|-1.0     |1            |0            |NULL       |
|-1.0     |1            |0            |NULL       |
|-1.0     |1            |0            |NULL       |
|-1.0     |1            |0            |NULL       |
|-1.0     |1            |0            |NULL       |
|-1.0     |1            |0            |NULL       |
|-1.0     |1            |0            |NULL      

In [108]:
df = df.withColumn(
    "性別_二元",
    F.when(F.col("性別") == "男", 1)
     .when(F.col("性別") == "女", 0)
     .otherwise(0)
)

df = df.withColumn(
    "婚姻狀況_二元",
    F.when(F.col("婚姻狀況") == "已婚", 1)
     .when(F.col("婚姻狀況") == "未婚", 0)
     .otherwise(0)
)


In [109]:
df = df.withColumn(
    "教育程度_序位",
    F.when(F.col("教育程度") == "Missing", 0)
     .when(F.col("教育程度") == "其他", 1)
     .when(F.col("教育程度") == "國中", 2)
     .when(F.col("教育程度") == "高中", 3)
     .when(F.col("教育程度") == "專科", 4)
     .when(F.col("教育程度") == "大學", 5)
     .when(F.col("教育程度") == "碩士以上", 6)
     .otherwise(0)
)

In [110]:
# 月所得處理：用序位編碼 0-14
# 0 = Missing，1 = 最低，14 = 最高
# 粗分區間給中間序位

df = df.withColumn(
    "月所得_序位",
    F.when(F.col("月所得") == "Missing", 0)
     # 細分區間 (1-14)
     .when(F.col("月所得") == "~20,000", 1)
     .when(F.col("月所得") == "20,000~24,999", 2)
     .when(F.col("月所得") == "25,000~29,999", 3)
     .when(F.col("月所得") == "30,000~34,999", 4)
     .when(F.col("月所得") == "35,000~39,999", 5)
     .when(F.col("月所得") == "40,000~44,999", 6)
     .when(F.col("月所得") == "45,000~49,999", 7)
     .when(F.col("月所得") == "50,000~54,999", 8)
     .when(F.col("月所得") == "55,000~59,999", 9)
     .when(F.col("月所得") == "60,000~64,999", 10)
     .when(F.col("月所得") == "65,000~69,999", 11)
     .when(F.col("月所得") == "70,000~74,999", 12)
     .when(F.col("月所得") == "75,000~79,999", 13)
     .when(F.col("月所得") == "80,000~", 14)
     # 粗分區間 → 給中間序位
     .when(F.col("月所得") == "20,000~29,999", 2)      # 對應 20k-30k 中間
     .when(F.col("月所得") == "30,000~39,999", 4)      # 對應 30k-40k 中間
     .when(F.col("月所得") == "40,000~49,999", 6)      # 對應 40k-50k 中間
     .when(F.col("月所得") == "50,000~59,999", 8)      # 對應 50k-60k 中間
     .when(F.col("月所得") == "60,000~69,999", 10)     # 對應 60k-70k 中間
     .when(F.col("月所得") == "70,000~79,999", 12)     # 對應 70k-80k 中間
     .otherwise(0)
)

# 缺失旗標
df = df.withColumn(
    "月所得_是否缺失",
    F.when(F.col("月所得") == "Missing", 1).otherwise(0)
)

# 檢查轉換結果
print("月所得序位編碼結果：")
df.groupBy("月所得", "月所得_序位").count().orderBy("月所得_序位").show(30, truncate=False)

月所得序位編碼結果：
+-------------+-----------+-----+
|月所得       |月所得_序位|count|
+-------------+-----------+-----+
|Missing      |0          |4091 |
|20,000以下   |0          |4473 |
|20,000~29,999|2          |241  |
|20,000~24,999|2          |26705|
|25,000~29,999|3          |8313 |
|30,000~39,999|4          |145  |
|30,000~34,999|4          |74544|
|35,000~39,999|5          |14472|
|40,000~49,999|6          |54   |
|40,000~44,999|6          |14668|
|45,000~49,999|7          |8041 |
|50,000~54,999|8          |10832|
|50,000~59,999|8          |27   |
|55,000~59,999|9          |2656 |
|60,000~69,999|10         |9    |
|60,000~64,999|10         |5296 |
|65,000~69,999|11         |1288 |
|70,000~74,999|12         |2327 |
|70,000~79,999|12         |5    |
|75,000~79,999|13         |654  |
|80,000~      |14         |5210 |
+-------------+-----------+-----+

+-------------+-----------+-----+
|月所得       |月所得_序位|count|
+-------------+-----------+-----+
|Missing      |0          |4091 |
|20,000以下   |0      

In [111]:
df = df.withColumn(
    "年齡組",
    F.when(F.col("年齡") < 21, "~20")
     .when((F.col("年齡") >= 21) & (F.col("年齡") < 31), "21-30")
     .when((F.col("年齡") >= 31) & (F.col("年齡") < 41), "31-40")
     .when((F.col("年齡") >= 41) & (F.col("年齡") < 51), "41-50")
     .when((F.col("年齡") >= 51) & (F.col("年齡") < 61), "51-60")
     .otherwise("61~")
)

In [112]:
df = df.withColumn(
    "年齡組_序位",
    F.when(F.col("年齡組") == "~20", 0)
     .when(F.col("年齡組") == "21-30", 1)
     .when(F.col("年齡組") == "31-40", 2)
     .when(F.col("年齡組") == "41-50", 3)
     .when(F.col("年齡組") == "51-60", 4)
     .when(F.col("年齡組") == "61~", 5)
     .otherwise(None)
)

In [113]:
# 內部往來次數處理（-1 為特殊值/缺失）
df = df.withColumn(
    "內部往來次數_是否特殊值",
    F.when(F.col("內部往來次數") == -1, 1).otherwise(0)
)

df = df.withColumn(
    "內部往來次數_清理後",
    F.when(F.col("內部往來次數") < 0, 0).otherwise(F.col("內部往來次數"))
)

# 近半年同業查詢次數處理（-1 也代表缺失）
df = df.withColumn(
    "近半年同業查詢次數_是否缺失",
    F.when(F.col("近半年同業查詢次數") == -1, 1).otherwise(0)
)

df = df.withColumn(
    "近半年同業查詢次數_清理後",
    F.when(F.col("近半年同業查詢次數") < 0, 0).otherwise(F.col("近半年同業查詢次數"))
)

# 所留市內電話數處理
df = df.withColumn(
    "所留市內電話數_清理後",
    F.when(F.col("所留市內電話數") < 0, 0).otherwise(F.col("所留市內電話數"))
)

In [114]:
# 檢查數值欄位的 Skewness（偏態）
# Skewness > 1 或 < -1 表示高度偏態，適合 log transform

from pyspark.sql.functions import skewness, mean, stddev, min as spark_min, max as spark_max

numeric_cols_to_check = [
    "原申辦金額",
    "年齡", 
    "車齡_清理後",
    "內部往來次數_清理後",
    "近半年同業查詢次數_清理後",
    "所留市內電話數_清理後"
]

print("數值欄位偏態分析（Skewness Analysis）：")
print("=" * 80)
print("| Skewness | > 1 或 < -1 為高度偏態，建議 log transform")
print("=" * 80)

for col in numeric_cols_to_check:
    stats = df.select(
        F.lit(col).alias("column"),
        skewness(F.col(col)).alias("skewness"),
        mean(F.col(col)).alias("mean"),
        stddev(F.col(col)).alias("stddev"),
        spark_min(F.col(col)).alias("min"),
        spark_max(F.col(col)).alias("max")
    ).collect()[0]
    
    skew_val = stats["skewness"]
    need_log = abs(skew_val) > 1 if skew_val is not None else False
    
    print(f"\n{col}:")
    print(f"  Skewness: {skew_val:.4f}" if skew_val else f"  Skewness: NULL")
    print(f"  Mean: {stats['mean']:.2f}" if stats['mean'] else f"  Mean: NULL")
    print(f"  Std: {stats['stddev']:.2f}" if stats['stddev'] else f"  Std: NULL")
    print(f"  Range: [{stats['min']}, {stats['max']}]")
    print(f"  → {'需要 Log Transform ✓' if need_log else '不需要 Log Transform'}")

數值欄位偏態分析（Skewness Analysis）：
| Skewness | > 1 或 < -1 為高度偏態，建議 log transform

原申辦金額:
  Skewness: 2.9264
  Mean: 90338.68
  Std: 24560.13
  Range: [6000.0, 1470000.0]
  → 需要 Log Transform ✓

原申辦金額:
  Skewness: 2.9264
  Mean: 90338.68
  Std: 24560.13
  Range: [6000.0, 1470000.0]
  → 需要 Log Transform ✓

年齡:
  Skewness: 0.3237
  Mean: 36.99
  Std: 13.99
  Range: [0, 88]
  → 不需要 Log Transform

年齡:
  Skewness: 0.3237
  Mean: 36.99
  Std: 13.99
  Range: [0, 88]
  → 不需要 Log Transform

車齡_清理後:
  Skewness: 2.2137
  Mean: 1.77
  Std: 3.23
  Range: [0.0, 38.02]
  → 需要 Log Transform ✓

車齡_清理後:
  Skewness: 2.2137
  Mean: 1.77
  Std: 3.23
  Range: [0.0, 38.02]
  → 需要 Log Transform ✓

內部往來次數_清理後:
  Skewness: 14.1604
  Mean: 0.28
  Std: 0.75
  Range: [0, 67]
  → 需要 Log Transform ✓

內部往來次數_清理後:
  Skewness: 14.1604
  Mean: 0.28
  Std: 0.75
  Range: [0, 67]
  → 需要 Log Transform ✓

近半年同業查詢次數_清理後:
  Skewness: 10.1085
  Mean: 1.26
  Std: 1.10
  Range: [0, 40]
  → 需要 Log Transform ✓

近半年同業查詢次數_清理後:
  Skewness:

In [115]:
# Log Transform（針對右偏分佈的欄位）
# log1p(x) = log(1 + x)，可處理 0 值

# 金額類 - 通常高度右偏
df = df.withColumn("原申辦金額_log", F.log1p(F.col("原申辦金額")))

# 車齡 - 缺失值填 0 後取 log
df = df.withColumn("車齡_log", F.log1p(F.coalesce(F.col("車齡_清理後"), F.lit(0.0))))

# 次數類 - 通常右偏（大部分是小值，少數大值）
df = df.withColumn("內部往來次數_log", F.log1p(F.col("內部往來次數_清理後")))
df = df.withColumn("近半年同業查詢次數_log", F.log1p(F.col("近半年同業查詢次數_清理後")))
df = df.withColumn("所留市內電話數_log", F.log1p(F.col("所留市內電話數_清理後")))

print("Log Transform 完成！")
print("新增欄位：原申辦金額_log, 車齡_log, 內部往來次數_log, 近半年同業查詢次數_log, 所留市內電話數_log")

Log Transform 完成！
新增欄位：原申辦金額_log, 車齡_log, 內部往來次數_log, 近半年同業查詢次數_log, 所留市內電話數_log


In [116]:
df = df.withColumn(
    "原申辦金額_數值",
    F.regexp_replace(F.col("原申辦金額").cast("string"), ",", "").cast(DoubleType())
)

In [117]:
if "成功案例" in df.columns:
    df = df.withColumn("成功案例_是否排除", F.lit(1))
else:
    df = df.withColumn("成功案例_是否排除", F.lit(None))

In [118]:
# ============================================
# 儲存 Silver Layer
# ============================================

# 新增 Silver 處理時間戳記
df = df.withColumn("silver_process_timestamp", F.current_timestamp())

# 檢查最終欄位
print("Silver Layer 最終欄位：")
print(f"總欄位數: {len(df.columns)}")
print(f"總筆數: {df.count()}")
print("\n欄位列表：")
for i, col in enumerate(sorted(df.columns), 1):
    print(f"  {i:2d}. {col}")

# 顯示 schema
df.printSchema()

Silver Layer 最終欄位：
總欄位數: 48
總筆數: 184051

欄位列表：
   1. bronze_load_timestamp
   2. silver_process_timestamp
   3. 內部往來次數
   4. 內部往來次數_log
   5. 內部往來次數_是否特殊值
   6. 內部往來次數_清理後
   7. 動產設定
   8. 動產設定_二元
   9. 原申辦金額
  10. 原申辦金額_log
  11. 原申辦金額_數值
  12. 婚姻狀況
  13. 婚姻狀況_二元
  14. 居住地
  15. 年齡
  16. 年齡組
  17. 年齡組_序位
  18. 廠牌車型
  19. 性別
  20. 性別_二元
  21. 成功案例
  22. 成功案例_是否排除
  23. 所留市內電話數
  24. 所留市內電話數_log
  25. 所留市內電話數_清理後
  26. 授信結果
  27. 授信結果_二元
  28. 教育程度
  29. 教育程度_序位
  30. 教育程度_是否缺失
  31. 教育程度_補值後
  32. 月所得
  33. 月所得_序位
  34. 月所得_是否缺失
  35. 案件編號
  36. 申辦期數
  37. 職業說明
  38. 車齡
  39. 車齡_log
  40. 車齡_原始
  41. 車齡_是否缺失
  42. 車齡_清理後
  43. 車齡_異常旗標
  44. 近半年同業查詢次數
  45. 近半年同業查詢次數_log
  46. 近半年同業查詢次數_是否缺失
  47. 近半年同業查詢次數_清理後
  48. 進件日
root
 |-- 案件編號: string (nullable = true)
 |-- 進件日: date (nullable = true)
 |-- 申辦期數: integer (nullable = true)
 |-- 原申辦金額: double (nullable = true)
 |-- 性別: string (nullable = false)
 |-- 年齡: integer (nullable = true)
 |-- 教育程度: string (nullable = false)
 |-- 婚姻狀況: stri

In [119]:
# 寫入 Silver Parquet
df.write.mode("overwrite").parquet(str(silver_output_path))

print(f"✓ Silver Layer 儲存完成！")
print(f"  路徑: {silver_output_path}")
print(f"  筆數: {df.count()}")

# 驗證寫入結果
df_silver_check = spark.read.parquet(str(silver_output_path))
print(f"  驗證讀取筆數: {df_silver_check.count()}")

✓ Silver Layer 儲存完成！
  路徑: c:\Users\88696\Desktop\ml_project\data\silver\silver_application_clean.parquet
  筆數: 184051
  筆數: 184051
  驗證讀取筆數: 184051
  驗證讀取筆數: 184051


In [120]:
# ============================================
# Gold Layer: 讀取 Silver 資料
# ============================================

df_gold = spark.read.parquet(str(silver_output_path))

print(f"Silver 資料筆數: {df_gold.count()}")
print(f"日期範圍:")
df_gold.select(
    F.min("進件日").alias("min_date"),
    F.max("進件日").alias("max_date")
).show()

# 新增年月欄位，方便後續切割
df_gold = df_gold.withColumn("進件年月", F.date_format(F.col("進件日"), "yyyy-MM"))

Silver 資料筆數: 184051
日期範圍:
+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2024-04-01|2026-03-31|
+----------+----------+

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2024-04-01|2026-03-31|
+----------+----------+



In [121]:
# ============================================
# Dev / OOT 切分
# ============================================
# 資料範圍：2024-04-01 ~ 2026-03-31（24個月）
# Development: 前18個月 (2024-04 ~ 2025-09)
# OOT: 後6個月 (2025-10 ~ 2026-03)

from datetime import date
from dateutil.relativedelta import relativedelta

# 計算切分點
data_start = date(2024, 4, 1)
data_end = date(2026, 3, 31)
oot_start = date(2025, 10, 1)  # OOT 從 2025-10 開始

print(f"資料範圍: {data_start} ~ {data_end}")
print(f"Development: {data_start} ~ {oot_start - relativedelta(days=1)}")
print(f"OOT: {oot_start} ~ {data_end}")

# 切分
df_dev = df_gold.filter(F.col("進件日") < F.lit(oot_start))
df_oot = df_gold.filter(F.col("進件日") >= F.lit(oot_start))

print(f"\nDevelopment 筆數: {df_dev.count()}")
print(f"OOT 筆數: {df_oot.count()}")

# 檢查 label 分布
print("\n=== Development Label 分布 ===")
df_dev.groupBy("授信結果_二元").count().show()

print("=== OOT Label 分布 ===")
df_oot.groupBy("授信結果_二元").count().show()

資料範圍: 2024-04-01 ~ 2026-03-31
Development: 2024-04-01 ~ 2025-09-30
OOT: 2025-10-01 ~ 2026-03-31

Development 筆數: 136696

Development 筆數: 136696
OOT 筆數: 47355

=== Development Label 分布 ===
OOT 筆數: 47355

=== Development Label 分布 ===
+-------------+------+
|授信結果_二元| count|
+-------------+------+
|            1|130369|
|            0|  6327|
+-------------+------+

=== OOT Label 分布 ===
+-------------+------+
|授信結果_二元| count|
+-------------+------+
|            1|130369|
|            0|  6327|
+-------------+------+

=== OOT Label 分布 ===
+-------------+-----+
|授信結果_二元|count|
+-------------+-----+
|            1|45094|
|            0| 2261|
+-------------+-----+

+-------------+-----+
|授信結果_二元|count|
+-------------+-----+
|            1|45094|
|            0| 2261|
+-------------+-----+



In [122]:
# ============================================
# Rolling Windows 定義
# ============================================
# 每個 window: 4個月訓練 + 2個月監控
# 從 2024-04 開始，每2個月滾動一次

rolling_windows = []

train_months = 4
monitor_months = 2
step_months = 2

current_start = data_start
window_id = 1

while True:
    train_end = current_start + relativedelta(months=train_months) - relativedelta(days=1)
    monitor_start = train_end + relativedelta(days=1)
    monitor_end = monitor_start + relativedelta(months=monitor_months) - relativedelta(days=1)
    
    # 如果監控期超過 OOT 開始日，就停止
    if monitor_end >= oot_start:
        break
    
    rolling_windows.append({
        "window_id": window_id,
        "train_start": current_start,
        "train_end": train_end,
        "monitor_start": monitor_start,
        "monitor_end": monitor_end
    })
    
    current_start = current_start + relativedelta(months=step_months)
    window_id += 1

# 顯示所有 rolling windows
print(f"總共 {len(rolling_windows)} 個 Rolling Windows:\n")
for w in rolling_windows:
    print(f"Window {w['window_id']}:")
    print(f"  Train:   {w['train_start']} ~ {w['train_end']}")
    print(f"  Monitor: {w['monitor_start']} ~ {w['monitor_end']}")
    print()

總共 7 個 Rolling Windows:

Window 1:
  Train:   2024-04-01 ~ 2024-07-31
  Monitor: 2024-08-01 ~ 2024-09-30

Window 2:
  Train:   2024-06-01 ~ 2024-09-30
  Monitor: 2024-10-01 ~ 2024-11-30

Window 3:
  Train:   2024-08-01 ~ 2024-11-30
  Monitor: 2024-12-01 ~ 2025-01-31

Window 4:
  Train:   2024-10-01 ~ 2025-01-31
  Monitor: 2025-02-01 ~ 2025-03-31

Window 5:
  Train:   2024-12-01 ~ 2025-03-31
  Monitor: 2025-04-01 ~ 2025-05-31

Window 6:
  Train:   2025-02-01 ~ 2025-05-31
  Monitor: 2025-06-01 ~ 2025-07-31

Window 7:
  Train:   2025-04-01 ~ 2025-07-31
  Monitor: 2025-08-01 ~ 2025-09-30



In [ ]:
# ============================================
# Model-ready 特徵定義
# ============================================
# 原則：只保留最終會進模型的欄位
# - 數值欄位：只保留 _log 版本（會再做 MinMaxScaler）
# - 序位欄位：直接使用（0-6, 0-14 已經是合理範圍）
# - 二元欄位：直接使用（0/1 不需要 scale）

# 數值特徵（需要 MinMaxScaler 的）
# 選用 log 版本，因為原始值偏態嚴重
numeric_features_to_scale = [
    "年齡",                         # 連續值，範圍合理
    "申辦期數",                     # 連續值
    "原申辦金額_log",               # 用 log 版本
    "車齡_log",                     # 用 log 版本
    "內部往來次數_log",             # 用 log 版本
    "近半年同業查詢次數_log",       # 用 log 版本
    "所留市內電話數_log",           # 用 log 版本
]

# 序位特徵（已編碼，不需要 scale）
ordinal_features = [
    "教育程度_序位",    # 0-6
    "月所得_序位",      # 0-14
    "年齡組_序位",      # 0-5
]

# 二元特徵（0/1，不需要 scale）
binary_features = [
    "性別_二元",
    "婚姻狀況_二元",
    "動產設定",                     # 原始資料已經是 0/1
    "車齡_是否缺失",
    "車齡_異常旗標",
    "教育程度_是否缺失",
    "月所得_是否缺失",
    "內部往來次數_是否特殊值",
    "近半年同業查詢次數_是否缺失",
]

# 高基數類別（用 Frequency Encoding）
high_cardinality_features = [
    "居住地",
    "職業說明",
    "廠牌車型",
]

# 目標變數
target_col = "授信結果_二元"

# Key 欄位（不進模型，但保留追蹤用）
key_cols = ["案件編號", "進件日", "進件年月"]

print("特徵定義完成：")
print(f"  數值特徵（需 scale）: {len(numeric_features_to_scale)} 個")
print(f"  序位特徵: {len(ordinal_features)} 個")
print(f"  二元特徵: {len(binary_features)} 個")
print(f"  高基數類別（Freq Encoding）: {len(high_cardinality_features)} 個")
print(f"  目標變數: {target_col}")

特徵定義完成：
  數值特徵: 12 個
  序位特徵: 3 個
  二元特徵: 9 個
  高基數類別: 3 個（之後用 Target Encoding）
  目標變數: 授信結果_二元


In [124]:
# ============================================
# 高基數類別處理：Frequency Encoding
# ============================================
# 對於類別太多的欄位，用出現頻率來編碼
# 這樣不會有 data leakage，因為只用 count

for col in high_cardinality_features:
    if col in df_dev.columns:
        # 計算 development set 中每個類別的頻率
        freq_df = df_dev.groupBy(col).count()
        total = df_dev.count()
        freq_df = freq_df.withColumn(
            f"{col}_頻率",
            F.col("count") / F.lit(total)
        ).select(col, f"{col}_頻率")
        
        # Join 回 df_dev 和 df_oot
        df_dev = df_dev.join(freq_df, on=col, how="left")
        df_oot = df_oot.join(freq_df, on=col, how="left")
        
        # OOT 中未見過的類別，頻率設為 0
        df_oot = df_oot.withColumn(
            f"{col}_頻率",
            F.coalesce(F.col(f"{col}_頻率"), F.lit(0.0))
        )
        
        print(f"✓ {col} 完成 Frequency Encoding")

# 更新特徵列表
frequency_features = [f"{col}_頻率" for col in high_cardinality_features]
print(f"\n新增頻率特徵: {frequency_features}")

✓ 居住地 完成 Frequency Encoding
✓ 職業說明 完成 Frequency Encoding
✓ 職業說明 完成 Frequency Encoding
✓ 廠牌車型 完成 Frequency Encoding

新增頻率特徵: ['居住地_頻率', '職業說明_頻率', '廠牌車型_頻率']
✓ 廠牌車型 完成 Frequency Encoding

新增頻率特徵: ['居住地_頻率', '職業說明_頻率', '廠牌車型_頻率']


In [ ]:
# ============================================
# Cross Features 交互特徵
# ============================================
# 注意：交互特徵如果任一輸入是 NULL，結果也會是 NULL
# 這裡先將可能是 NULL 的欄位填補為 0，因為：
# 1. 我們已經有對應的缺失旗標（車齡_是否缺失 等）
# 2. 模型可以透過旗標學習「這個交互特徵是填補值」

# 1. 負債收入比（申辦金額 / 月所得序位）
df_dev = df_dev.withColumn(
    "申辦金額_月所得比",
    F.when(F.col("月所得_序位") > 0, 
           F.col("原申辦金額") / F.col("月所得_序位"))
     .otherwise(F.col("原申辦金額"))
)

df_oot = df_oot.withColumn(
    "申辦金額_月所得比",
    F.when(F.col("月所得_序位") > 0, 
           F.col("原申辦金額") / F.col("月所得_序位"))
     .otherwise(F.col("原申辦金額"))
)

# 2. 年齡 x 婚姻狀況
df_dev = df_dev.withColumn("年齡_婚姻交互", F.col("年齡") * F.col("婚姻狀況_二元"))
df_oot = df_oot.withColumn("年齡_婚姻交互", F.col("年齡") * F.col("婚姻狀況_二元"))

# 3. 車齡 x 申辦金額
# 車齡_清理後 可能是 NULL（缺失或異常），用 0 填補
# 因為我們已有 車齡_是否缺失 和 車齡_異常旗標 來標記這些情況
df_dev = df_dev.withColumn(
    "車齡_金額交互",
    F.coalesce(F.col("車齡_清理後"), F.lit(0.0)) * F.col("原申辦金額_log")
)
df_oot = df_oot.withColumn(
    "車齡_金額交互",
    F.coalesce(F.col("車齡_清理後"), F.lit(0.0)) * F.col("原申辦金額_log")
)

# 4. 教育程度 x 月所得
df_dev = df_dev.withColumn("教育_所得交互", F.col("教育程度_序位") * F.col("月所得_序位"))
df_oot = df_oot.withColumn("教育_所得交互", F.col("教育程度_序位") * F.col("月所得_序位"))

cross_features = [
    "申辦金額_月所得比",
    "年齡_婚姻交互",
    "車齡_金額交互",
    "教育_所得交互"
]

print(f"✓ 新增交互特徵: {cross_features}")
print("  註：車齡_金額交互 中，車齡缺失時以 0 計算（有旗標標記）")

✓ 新增交互特徵: ['申辦金額_月所得比', '年齡_婚姻交互', '車齡_金額交互', '教育_所得交互']


In [ ]:
# ============================================
# MinMaxScaler 標準化
# ============================================
# 只對數值特徵 + 交互特徵 + 頻率特徵做 MinMax 標準化
# 用 Development set 計算 min/max，套用到 OOT

# 要標準化的欄位
scale_cols = numeric_features_to_scale + cross_features + frequency_features

# 計算 Development 的 min/max
scale_stats = {}
for col in scale_cols:
    if col in df_dev.columns:
        stats = df_dev.select(
            F.min(col).alias("min_val"),
            F.max(col).alias("max_val")
        ).collect()[0]
        scale_stats[col] = {
            "min": stats["min_val"] if stats["min_val"] is not None else 0,
            "max": stats["max_val"] if stats["max_val"] is not None else 1
        }

# 套用 MinMax 標準化
for col in scale_cols:
    if col in scale_stats and col in df_dev.columns:
        min_val = scale_stats[col]["min"]
        max_val = scale_stats[col]["max"]
        range_val = max_val - min_val if max_val != min_val else 1
        
        scaled_col = f"{col}_scaled"
        
        df_dev = df_dev.withColumn(
            scaled_col,
            (F.col(col) - F.lit(min_val)) / F.lit(range_val)
        )
        
        df_oot = df_oot.withColumn(
            scaled_col,
            (F.col(col) - F.lit(min_val)) / F.lit(range_val)
        )

scaled_features = [f"{col}_scaled" for col in scale_cols if col in df_dev.columns]
print(f"✓ MinMaxScaler 完成，共 {len(scaled_features)} 個標準化特徵")

✓ MinMaxScaler 完成，共 19 個標準化特徵


In [ ]:
# ============================================
# 最終 Model Input 欄位
# ============================================
# Gold 層只輸出：Key + 最終特徵 + Target
# 不輸出中間處理欄位（原始值、未 scale 的值等）

# 組合所有特徵
all_model_features = (
    ordinal_features +      # 序位編碼（直接使用）
    binary_features +       # 二元特徵（直接使用）
    scaled_features         # 標準化後的數值特徵
)

# ============================================
# 防禦性 NULL 處理
# ============================================
# 這裡用 0 填補 NULL 是安全的，因為：
# 1. 序位特徵：Missing 已經編碼為 0，不會有 NULL
# 2. 二元特徵：都是 0/1，已經處理過
# 3. scaled 特徵：
#    - 數值欄位：已經在 Silver 層清理過（車齡缺失有旗標）
#    - 交互特徵：已經在建立時用 coalesce 處理
#    - 頻率特徵：OOT 未見類別已設為 0
# 
# 這只是最後一道防線，理論上不應該有 NULL

# 檢查是否真的有 NULL（Debug 用）
print("檢查 NULL 數量：")
null_count_dev = 0
null_count_oot = 0
for col in all_model_features:
    if col in df_dev.columns:
        cnt = df_dev.filter(F.col(col).isNull()).count()
        if cnt > 0:
            print(f"  [Dev] {col}: {cnt} NULLs")
            null_count_dev += cnt
    if col in df_oot.columns:
        cnt = df_oot.filter(F.col(col).isNull()).count()
        if cnt > 0:
            print(f"  [OOT] {col}: {cnt} NULLs")
            null_count_oot += cnt

if null_count_dev == 0 and null_count_oot == 0:
    print("  ✓ 所有特徵都沒有 NULL，不需要填補！")
else:
    print(f"  ⚠ Dev 共 {null_count_dev} 個 NULL，OOT 共 {null_count_oot} 個 NULL")
    print("  → 以下進行填補...")
    for col in all_model_features:
        if col in df_dev.columns:
            df_dev = df_dev.withColumn(col, F.coalesce(F.col(col), F.lit(0.0)))
        if col in df_oot.columns:
            df_oot = df_oot.withColumn(col, F.coalesce(F.col(col), F.lit(0.0)))

# 最終欄位：key + features + target
final_cols = key_cols + all_model_features + [target_col]

# 只保留需要的欄位
df_dev_final = df_dev.select(*[c for c in final_cols if c in df_dev.columns])
df_oot_final = df_oot.select(*[c for c in final_cols if c in df_oot.columns])

print("\n" + "=" * 60)
print("Gold Model-Ready 資料準備完成")
print("=" * 60)

print(f"\n【欄位組成】")
print(f"  Key 欄位: {len(key_cols)} 個 - {key_cols}")
print(f"  序位特徵: {len(ordinal_features)} 個")
print(f"  二元特徵: {len(binary_features)} 個")
print(f"  標準化特徵: {len(scaled_features)} 個")
print(f"  目標變數: 1 個 - {target_col}")
print(f"  ─────────────────────────")
print(f"  總欄位數: {len(final_cols)} 個")

print(f"\n【資料集統計】")
print(f"  Development: {df_dev_final.count()} 筆")
print(f"  OOT: {df_oot_final.count()} 筆")

print(f"\n【模型特徵列表】（共 {len(all_model_features)} 個）")
for i, f in enumerate(all_model_features, 1):
    print(f"  {i:2d}. {f}")

Gold Model-Ready 資料準備完成

Development Set:
  筆數: 136696
  欄位數: 35

OOT Set:
  筆數: 47355
  欄位數: 35

模型特徵數: 31

特徵列表：
   1. 教育程度_序位
   2. 月所得_序位
   3. 年齡組_序位
   4. 性別_二元
   5. 婚姻狀況_二元
   6. 動產設定_二元
   7. 車齡_是否缺失
   8. 車齡_異常旗標
   9. 教育程度_是否缺失
  10. 月所得_是否缺失
  11. 內部往來次數_是否特殊值
  12. 近半年同業查詢次數_是否缺失
  13. 年齡_scaled
  14. 申辦期數_scaled
  15. 原申辦金額_scaled
  16. 車齡_清理後_scaled
  17. 內部往來次數_清理後_scaled
  18. 近半年同業查詢次數_清理後_scaled
  19. 所留市內電話數_清理後_scaled
  20. 原申辦金額_log_scaled
  21. 車齡_log_scaled
  22. 內部往來次數_log_scaled
  23. 近半年同業查詢次數_log_scaled
  24. 所留市內電話數_log_scaled
  25. 申辦金額_月所得比_scaled
  26. 年齡_婚姻交互_scaled
  27. 車齡_金額交互_scaled
  28. 教育_所得交互_scaled
  29. 居住地_頻率_scaled
  30. 職業說明_頻率_scaled
  31. 廠牌車型_頻率_scaled
  筆數: 136696
  欄位數: 35

OOT Set:
  筆數: 47355
  欄位數: 35

模型特徵數: 31

特徵列表：
   1. 教育程度_序位
   2. 月所得_序位
   3. 年齡組_序位
   4. 性別_二元
   5. 婚姻狀況_二元
   6. 動產設定_二元
   7. 車齡_是否缺失
   8. 車齡_異常旗標
   9. 教育程度_是否缺失
  10. 月所得_是否缺失
  11. 內部往來次數_是否特殊值
  12. 近半年同業查詢次數_是否缺失
  13. 年齡_scaled
  14. 申辦期數_scaled
  15

In [128]:
# ============================================
# 以月份切割輸出 Gold Parquet
# ============================================

# 建立輸出目錄
gold_monthly_path = project_root / "data" / "gold" / "monthly"

# Development：按月份切割
print("儲存 Development 資料（按月份）...")
df_dev_final.write.mode("overwrite").partitionBy("進件年月").parquet(
    str(gold_monthly_path / "development")
)
print(f"✓ Development 儲存至: {gold_monthly_path / 'development'}")

# OOT：按月份切割
print("\n儲存 OOT 資料（按月份）...")
df_oot_final.write.mode("overwrite").partitionBy("進件年月").parquet(
    str(gold_monthly_path / "oot")
)
print(f"✓ OOT 儲存至: {gold_monthly_path / 'oot'}")

# 顯示月份分布
print("\n=== Development 月份分布 ===")
df_dev_final.groupBy("進件年月").count().orderBy("進件年月").show(24, truncate=False)

print("=== OOT 月份分布 ===")
df_oot_final.groupBy("進件年月").count().orderBy("進件年月").show(12, truncate=False)

儲存 Development 資料（按月份）...
✓ Development 儲存至: c:\Users\88696\Desktop\ml_project\data\gold\monthly\development

儲存 OOT 資料（按月份）...
✓ Development 儲存至: c:\Users\88696\Desktop\ml_project\data\gold\monthly\development

儲存 OOT 資料（按月份）...
✓ OOT 儲存至: c:\Users\88696\Desktop\ml_project\data\gold\monthly\oot

=== Development 月份分布 ===
✓ OOT 儲存至: c:\Users\88696\Desktop\ml_project\data\gold\monthly\oot

=== Development 月份分布 ===
+--------+-----+
|進件年月|count|
+--------+-----+
|2024-04 |6622 |
|2024-05 |8064 |
|2024-06 |7504 |
|2024-07 |8226 |
|2024-08 |7824 |
|2024-09 |7955 |
|2024-10 |6588 |
|2024-11 |6863 |
|2024-12 |7641 |
|2025-01 |6215 |
|2025-02 |8365 |
|2025-03 |7429 |
|2025-04 |6797 |
|2025-05 |7445 |
|2025-06 |7748 |
|2025-07 |8596 |
|2025-08 |8554 |
|2025-09 |8260 |
+--------+-----+

=== OOT 月份分布 ===
+--------+-----+
|進件年月|count|
+--------+-----+
|2024-04 |6622 |
|2024-05 |8064 |
|2024-06 |7504 |
|2024-07 |8226 |
|2024-08 |7824 |
|2024-09 |7955 |
|2024-10 |6588 |
|2024-11 |6863 |
|2024-12 |764

In [133]:
# ============================================
# 儲存 Rolling Windows 定義（用 CSV 格式）
# ============================================

import pandas as pd
import os

# 建立 Rolling Windows DataFrame
rolling_windows_pd = pd.DataFrame([
    {
        "window_id": w["window_id"],
        "train_start": str(w["train_start"]),
        "train_end": str(w["train_end"]),
        "monitor_start": str(w["monitor_start"]),
        "monitor_end": str(w["monitor_end"])
    }
    for w in rolling_windows
])

# 改用 CSV 儲存路徑
gold_window_def_csv = project_root / "data" / "gold" / "gold_rolling_window_definition.csv"

# 確保目錄存在
os.makedirs(gold_window_def_csv.parent, exist_ok=True)

# 用 CSV 格式儲存
rolling_windows_pd.to_csv(str(gold_window_def_csv), index=False, encoding="utf-8")
print(f"✓ Rolling Windows 定義儲存至: {gold_window_def_csv}")

# 顯示 Rolling Windows
print("\nRolling Windows 定義：")
print(rolling_windows_pd.to_string(index=False))

✓ Rolling Windows 定義儲存至: c:\Users\88696\Desktop\ml_project\data\gold\gold_rolling_window_definition.csv

Rolling Windows 定義：
 window_id train_start  train_end monitor_start monitor_end
         1  2024-04-01 2024-07-31    2024-08-01  2024-09-30
         2  2024-06-01 2024-09-30    2024-10-01  2024-11-30
         3  2024-08-01 2024-11-30    2024-12-01  2025-01-31
         4  2024-10-01 2025-01-31    2025-02-01  2025-03-31
         5  2024-12-01 2025-03-31    2025-04-01  2025-05-31
         6  2025-02-01 2025-05-31    2025-06-01  2025-07-31
         7  2025-04-01 2025-07-31    2025-08-01  2025-09-30


In [134]:
# ============================================
# 儲存完整的 Dev / OOT Parquet
# ============================================

# Development 完整資料
df_dev_final.write.mode("overwrite").parquet(str(gold_dev_path))
print(f"✓ Development 完整資料儲存至: {gold_dev_path}")

# OOT 完整資料
df_oot_final.write.mode("overwrite").parquet(str(gold_oot_path))
print(f"✓ OOT 完整資料儲存至: {gold_oot_path}")

print("\n" + "=" * 60)
print("🎉 Gold Layer 全部完成！")
print("=" * 60)
print(f"\n輸出檔案：")
print(f"  1. {gold_monthly_path / 'development'} (按月份)")
print(f"  2. {gold_monthly_path / 'oot'} (按月份)")
print(f"  3. {gold_dev_path} (完整)")
print(f"  4. {gold_oot_path} (完整)")
print(f"  5. {gold_window_def_path} (Rolling Windows 定義)")

✓ Development 完整資料儲存至: c:\Users\88696\Desktop\ml_project\data\gold\gold_development_dataset.parquet
✓ OOT 完整資料儲存至: c:\Users\88696\Desktop\ml_project\data\gold\gold_final_oot_dataset.parquet

🎉 Gold Layer 全部完成！

輸出檔案：
  1. c:\Users\88696\Desktop\ml_project\data\gold\monthly\development (按月份)
  2. c:\Users\88696\Desktop\ml_project\data\gold\monthly\oot (按月份)
  3. c:\Users\88696\Desktop\ml_project\data\gold\gold_development_dataset.parquet (完整)
  4. c:\Users\88696\Desktop\ml_project\data\gold\gold_final_oot_dataset.parquet (完整)
  5. c:\Users\88696\Desktop\ml_project\data\gold\gold_rolling_window_definition.parquet (Rolling Windows 定義)
✓ OOT 完整資料儲存至: c:\Users\88696\Desktop\ml_project\data\gold\gold_final_oot_dataset.parquet

🎉 Gold Layer 全部完成！

輸出檔案：
  1. c:\Users\88696\Desktop\ml_project\data\gold\monthly\development (按月份)
  2. c:\Users\88696\Desktop\ml_project\data\gold\monthly\oot (按月份)
  3. c:\Users\88696\Desktop\ml_project\data\gold\gold_development_dataset.parquet (完整)
  4. c:\Users\